# Training the legacy 1D CNN: CPU and GPU runs

This notebook reproduces the legacy 1D-CNN architecture, the original
row-wise maximum normalization, and the original dataset loading
contract. It trains one model per execution. Run it once with
`RUN_DEVICE = "cpu"` and once with `RUN_DEVICE = "cuda"`.

The saved wall-clock time covers the fixed 50-epoch train-and-validation
protocol. Dataset loading and output writing are outside the timer.


In [ ]:
from __future__ import annotations

import json
import platform
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd


WORK_ROOT = Path.cwd().resolve()

DATA_FOLDER = Path("/hercules/results/akazantsev/rfim_dataset")
META_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels_meta.csv"
SPLIT_PATH = DATA_FOLDER / "split_indices.npz"
PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_channels.npy"

# The training subset deliberately retains only statistical features and labels.
# These full files retain channel and segment identity and are used only by the
# inference-timing notebook, where one input must correspond to a real 256-channel
# observation segment.
FULL_META_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels_meta.csv"
FULL_PROFILES_PATH = DATA_FOLDER / "B0531+21_59000_48386_channels.npy"
SUBSET_SOURCE_INDICES_PATH = DATA_FOLDER / "B0531+21_59000_48386_subset_indices.npy"

# Change the tag only for a deliberate new experiment. Existing results are never overwritten.
RUN_TAG = "b0531_legacy_performance_v1"
RUN_ROOT = WORK_ROOT / "outputs" / "performance_comparison" / RUN_TAG


def json_ready(value):
    if isinstance(value, dict):
        return {key: json_ready(item) for key, item in value.items()}
    if isinstance(value, (np.integer, np.floating)):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    if isinstance(value, (list, tuple)):
        return [json_ready(item) for item in value]
    return value


def write_json(path: Path, payload: dict) -> None:
    with path.open("w", encoding="utf-8") as handle:
        json.dump(json_ready(payload), handle, indent=2, sort_keys=True)
        handle.write("\n")


def git_revision() -> str:
    try:
        return subprocess.check_output(
            ["git", "rev-parse", "HEAD"],
            cwd=WORK_ROOT,
            text=True,
            stderr=subprocess.DEVNULL,
        ).strip()
    except (FileNotFoundError, subprocess.CalledProcessError):
        return None


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset


In [ ]:
class CNN1DRFI256Logits(nn.Module):
    """Legacy 1D-CNN architecture for one 256-sample channel profile."""

    def __init__(self, dropout: float = 0.5):
        super().__init__()
        self.conv1 = nn.Conv1d(1, 64, kernel_size=7)
        self.conv2 = nn.Conv1d(64, 128, kernel_size=7)
        self.conv3 = nn.Conv1d(128, 256, kernel_size=10)
        self.dropout = nn.Dropout(dropout)
        self.fc1 = nn.Linear(256 * 25, 256)
        self.fc2 = nn.Linear(256, 1)

    def forward(self, x):
        x = F.max_pool1d(F.relu(self.conv1(x)), kernel_size=2)
        # The legacy architecture pads only this intermediate activation map.
        x = F.pad(x, (0, 1))
        x = F.max_pool1d(F.relu(self.conv2(x)), kernel_size=2)
        x = F.max_pool1d(F.relu(self.conv3(x)), kernel_size=2)
        x = x.flatten(start_dim=1)
        x = self.dropout(F.relu(self.fc1(x)))
        return self.fc2(x).squeeze(-1)


In [ ]:
RUN_DEVICE = "cpu"  # Run again with "cuda" after GPU is available.
BATCH_SIZE = 256
EPOCHS = 50
LEARNING_RATE = 1e-3
RANDOM_STATE = 42

if RUN_DEVICE not in {"cpu", "cuda"}:
    raise ValueError("RUN_DEVICE must be 'cpu' or 'cuda'.")
if RUN_DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("RUN_DEVICE='cuda', but CUDA is not available.")

device = torch.device(RUN_DEVICE)
output_dir = RUN_ROOT / f"cnn_{RUN_DEVICE}_legacy_max"
if output_dir.exists():
    raise FileExistsError(
        f"{output_dir} already exists. Choose a new RUN_TAG rather than overwrite it."
    )
output_dir.mkdir(parents=True)

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
if device.type == "cuda":
    torch.cuda.manual_seed_all(RANDOM_STATE)

print(f"Device: {device}")
print(f"Output: {output_dir}")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))


In [ ]:
# Keep the original paths and split keys used in the legacy CNN notebook.
meta = pd.read_csv(META_PATH).fillna("None")
profiles = np.load(PROFILES_PATH, mmap_mode="r")
splits = np.load(SPLIT_PATH)

train_idx = np.asarray(splits["train_idx"], dtype=int)
val_idx = np.asarray(splits["val_idx"], dtype=int)
test_idx = np.asarray(splits["test_idx"], dtype=int)
y = meta["label"].eq("NBRFI").to_numpy(dtype=np.int64)

if len(meta) != len(profiles) or len(y) != len(profiles):
    raise ValueError("Metadata, labels, and profile array must have identical row counts.")
if profiles.shape[1:] != (256,):
    raise ValueError(f"Expected profile rows with shape (256,), got {profiles.shape}.")

print("Profiles:", profiles.shape)
print("Class counts:", dict(zip(*np.unique(y, return_counts=True))))
print("Train / validation / test:", len(train_idx), len(val_idx), len(test_idx))


In [ ]:
class Simple1DDataset(Dataset):
    """Legacy dataset contract: one row, shape (1, 256), max normalization."""

    def __init__(self, values, labels, indices):
        self.values = values
        self.labels = labels
        self.indices = indices

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, position):
        row_index = self.indices[position]
        profile = np.asarray(self.values[row_index], dtype=np.float32)
        if profile.ndim == 1:
            profile = profile[None, :]
        elif profile.ndim == 2 and profile.shape[0] != 1:
            profile = profile.T

        maximum = float(profile.max())
        if maximum < 1e-8:
            maximum = 1.0
        profile = profile / maximum

        return (
            torch.from_numpy(profile),
            torch.tensor(self.labels[row_index], dtype=torch.float32),
        )


datasets = {
    "train": Simple1DDataset(profiles, y, train_idx),
    "validation": Simple1DDataset(profiles, y, val_idx),
    "test": Simple1DDataset(profiles, y, test_idx),
}
loaders = {
    "train": DataLoader(datasets["train"], batch_size=BATCH_SIZE, shuffle=True),
    "validation": DataLoader(datasets["validation"], batch_size=BATCH_SIZE, shuffle=False),
    "test": DataLoader(datasets["test"], batch_size=BATCH_SIZE, shuffle=False),
}

# Importing this class preserves the audited legacy architecture, including right padding.
model = CNN1DRFI256Logits(dropout=0.5).to(device)
loss_fn = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print(model)


In [ ]:
def run_epoch(loader, training: bool):
    model.train(training)
    loss_sum = 0.0
    correct = 0
    count = 0

    for batch_profiles, batch_labels in loader:
        batch_profiles = batch_profiles.to(device)
        batch_labels = batch_labels.to(device).reshape(-1)

        if training:
            optimizer.zero_grad(set_to_none=True)

        with torch.set_grad_enabled(training):
            logits = model(batch_profiles).reshape(-1)
            loss = loss_fn(logits, batch_labels)
            if training:
                loss.backward()
                optimizer.step()

        loss_sum += loss.item() * len(batch_labels)
        correct += ((torch.sigmoid(logits) >= 0.5) == batch_labels.bool()).sum().item()
        count += len(batch_labels)

    return {"loss": loss_sum / count, "accuracy": correct / count}


def synchronize_device():
    if device.type == "cuda":
        torch.cuda.synchronize(device)


checkpoint_path = output_dir / "checkpoint.pt"
history = []
best_validation_accuracy = -np.inf
best_epoch = None

synchronize_device()
training_started = time.perf_counter()

for epoch in range(1, EPOCHS + 1):
    epoch_started = time.perf_counter()
    train = run_epoch(loaders["train"], training=True)
    validation = run_epoch(loaders["validation"], training=False)
    synchronize_device()
    epoch_seconds = time.perf_counter() - epoch_started

    history.append({
        "epoch": epoch,
        "train_loss": train["loss"],
        "train_accuracy": train["accuracy"],
        "validation_loss": validation["loss"],
        "validation_accuracy": validation["accuracy"],
        "epoch_wall_clock_s": epoch_seconds,
    })
    print(
        f"Epoch {epoch:02d}/{EPOCHS}: "
        f"train loss={train['loss']:.5f}, acc={train['accuracy']:.4f}; "
        f"validation loss={validation['loss']:.5f}, acc={validation['accuracy']:.4f}; "
        f"{epoch_seconds:.2f} s"
    )

    if validation["accuracy"] > best_validation_accuracy:
        best_validation_accuracy = validation["accuracy"]
        best_epoch = epoch
        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "validation_accuracy": validation["accuracy"],
                "validation_loss": validation["loss"],
                "normalization": "legacy_max_per_channel",
            },
            checkpoint_path,
        )

synchronize_device()
training_wall_clock_s = time.perf_counter() - training_started
print(f"Training protocol finished in {training_wall_clock_s:.2f} s.")


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
test = run_epoch(loaders["test"], training=False)

pd.DataFrame(history).to_csv(output_dir / "history.csv", index=False)
summary = {
    "run_tag": RUN_TAG,
    "model": "CNN1DRFI256Logits",
    "architecture_source": "embedded legacy-compatible CNN1DRFI256Logits definition",
    "normalization": "legacy_max_per_channel",
    "device": str(device),
    "gpu_name": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
    "torch_version": torch.__version__,
    "python_version": platform.python_version(),
    "code_revision": git_revision(),
    "dataset": {
        "metadata_path": META_PATH,
        "profiles_path": PROFILES_PATH,
        "split_path": SPLIT_PATH,
        "n_train": len(train_idx),
        "n_validation": len(val_idx),
        "n_test": len(test_idx),
    },
    "training_protocol": {
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "optimizer": "Adam",
        "learning_rate": LEARNING_RATE,
        "loss": "BCEWithLogitsLoss",
        "checkpoint_selection": "maximum_validation_accuracy",
        "timer_scope": "fixed 50-epoch train-and-validation loop; excludes dataset loading and output writing",
    },
    "training_wall_clock_s": training_wall_clock_s,
    "best_epoch": best_epoch,
    "best_validation_accuracy": best_validation_accuracy,
    "test_loss": test["loss"],
    "test_accuracy_at_0_5": test["accuracy"],
    "artifacts": {"checkpoint": checkpoint_path, "history": output_dir / "history.csv"},
}
write_json(output_dir / "training_summary.json", summary)

print(json.dumps(json_ready(summary), indent=2))
